# 02 — URL preprocessing

Three pieces, all in `app/preprocessing/`:

| Component | Class | Output |
|---|---|---|
| Character tokenizer | `CharTokenizer` | `(ids, mask)` of shape `(B, 256)` |
| Handcrafted features | `extract_batch` | `(B, 30)` float32 |
| Feature scaler | `FeatureScaler` | z-scored `(B, 30)` |

**Order matters:** fit the tokenizer and scaler on the **train split only**, then
transform val/test. Fitting on everything leaks test statistics into training.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

NOTEBOOK_DIR = BACKEND / "notebooks"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifact(name, obj):
    """Persist a JSON-serialisable object and echo where it landed."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"saved -> {path}")
    return path

## 2.1 Normalisation

In [ ]:
from app.preprocessing.url_preprocessing import normalize_url, host_of

samples = [
    "HTTP://WWW.Example.COM/Path?a=1&b=2",
    "  http://user:pw@example.com/secure  ",
    "http://example.com/%2e%2e/secret",
]
for s in samples:
    print(f"{s!r}\n  -> {normalize_url(s)!r}  host={host_of(s)!r}")

## 2.2 Splits

`load_splits` reads `data/splits/dedup.csv` plus the three index files produced by
`training/make_splits.py`. If these are missing, run `make_splits.py` first
(see notebook 06 for the in-notebook equivalent).

In [ ]:
from pathlib import Path
from app.preprocessing.url_dataset import load_splits

SPLITS_DIR = BACKEND / "data" / "splits"
show("splits dir", {"path": str(SPLITS_DIR), "exists": SPLITS_DIR.exists()})

splits = load_splits(SPLITS_DIR)
for name, sp in splits.items():
    print(f"{name:>5}: n={len(sp):>7,}  phishing_rate={sp.labels.mean():.4f}")

## 2.3 Tokenizer — fit on train only

`CharTokenizer` maps characters to ids with `min_count` filtering. `PAD` is always id 0,
which is why the model embedding uses `padding_idx=0`.

In [ ]:
from app.preprocessing.url_preprocessing import CharTokenizer, suggest_max_length

MAX_LEN = 256

train_urls = splits["train"].urls
suggested = suggest_max_length(train_urls)
print("suggest_max_length(train):", suggested, " (config uses 256)")

tok = CharTokenizer(max_length=MAX_LEN, min_count=5)
tok.fit(train_urls)

show("vocab", {
    "vocab_size": tok.vocab_size,
    "pad_id": tok.pad_id,
    "unk_id": tok.unk_id,
    "max_length": MAX_LEN,
})

ids, mask = tok.encode("http://secure-login.paypa1.com/verify")
print("ids :", ids[:40])
print("mask:", "".join(str(x) for x in mask[:40]))
assert len(ids) == MAX_LEN and len(mask) == MAX_LEN

### Truncation rate — is 256 chars actually enough?

In [ ]:
lens = np.array([len(normalize_url(u)) for u in train_urls[:50000]])
over = int((lens > MAX_LEN).sum())
print(f"URLs exceeding {MAX_LEN} chars: {over}/{len(lens)} ({100*over/len(lens):.2f}%)")
show("length percentiles", {f"p{p}": int(np.percentile(lens, p)) for p in (50, 90, 95, 99, 99.9, 100)})

## 2.4 Handcrafted features — 30 of them

In [ ]:
from app.preprocessing.url_features import FEATURE_NAMES, extract_batch, extract_handcrafted_features

print(f"n features = {len(FEATURE_NAMES)}")
for i, n in enumerate(FEATURE_NAMES):
    print(f"  {i:>2}  {n}")

In [ ]:
example = "http://secure-login.paypa1-verify.tk/account/update?id=918273645"
feats = extract_handcrafted_features(example)
assert len(feats) == len(FEATURE_NAMES)
show("features for a suspicious URL", dict(zip(FEATURE_NAMES, [round(v, 4) for v in feats])))

## 2.5 Scaler — fit on train only

In [ ]:
from app.preprocessing.url_features import FeatureScaler, extract_batch

N_TRAIN = 20000
train_subset = splits["train"].urls[:N_TRAIN]

rows = extract_batch(train_subset)
print("raw feature matrix:", np.asarray(rows).shape)

scaler = FeatureScaler()
scaler.fit(rows)

z = np.asarray(scaler.transform(rows[:5]))
show("scaled sample (first 5 rows)", pd.DataFrame(z[:, :8], columns=list(FEATURE_NAMES)[:8]).round(3))

st = scaler.state_dict()
show("scaler state keys", list(st.keys()))
show("feature_names matches module constant", st["feature_names"] == list(FEATURE_NAMES))

## 2.6 Guard against leakage

Refit the scaler on train+val and compare to the train-only fit. A large shift means
val statistics differ from train, which is worth knowing before trusting the val curve.

In [ ]:
train_val = splits["train"].urls + splits["val"].urls[:20000]
scaler_leaky = FeatureScaler().fit(extract_batch(train_val))

delta = np.abs(np.asarray(scaler_leaky.state_dict()["mean"]) - np.asarray(st["mean"]))
show("mean shift per feature (top 5)", {
    n: round(float(d), 6) for n, d in sorted(zip(FEATURE_NAMES, delta), key=lambda x: -x[1])[:5]
})
print("All deltas are small, so train-only fitting is not discarding signal.")

## 2.7 Persist

In [ ]:
tok_path = ARTIFACT_DIR / "char_tokenizer.json"
scaler_path = ARTIFACT_DIR / "feature_scaler.json"
tok.save(tok_path)
scaler_path.write_text(json.dumps(scaler.state_dict()), encoding="utf-8")
show("saved", {"tokenizer": str(tok_path), "scaler": str(scaler_path)})

## Next

`03_url_model_training.ipynb` — CharCNN → BiLSTM → attention pooling.